# SanketVani (सङ्केतवाणी) — Continuous Sign Language Translator

Trains a character-level CTC translator on the
[iSign](https://huggingface.co/datasets/Exploration-Lab/iSign) MediaPipe-Holistic
keypoints and exports a browser-ready ONNX graph.

**Cell 1** streams the keypoint archive over HTTP Range requests straight into a
single compressed HDF5 file — no `.pose` file ever lands on disk, so Kaggle's
inode budget is never touched.
**Cell 2** trains `SignSentenceTranslator` (Conv1D -> 3x BiLSTM -> CTC) and
writes the three deliverables to `/kaggle/working/`.

Turn the notebook's internet switch **on** and pick a **GPU** accelerator, then
run the two cells in order.


In [1]:
# ── SanketVani full run: pre-flight + settings ──
import os, pathlib, subprocess, urllib.request

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("GPU      :", gpu or "NONE")
if not gpu:
    raise RuntimeError("No GPU attached. Settings > Accelerator > GPU, then commit again.")

urllib.request.urlopen("https://huggingface.co", timeout=20)   # raises if Internet is off
print("Internet : ON")

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN").strip()
    print("HF_TOKEN : found, %d chars" % len(os.environ["HF_TOKEN"]))
except Exception as e:
    print("HF_TOKEN : not attached, continuing anonymously --", e)

os.environ["SV_NUM_FRAMES"]    = "192"
os.environ["SV_TIME_UPSAMPLE"] = "2"
os.environ["SV_LIMIT_SAMPLES"] = "none"
os.environ["SV_EPOCHS"]        = "30"
pathlib.Path("/kaggle/working/isign_ctc_dataset.h5").unlink(missing_ok=True)

print("\nfull run armed: 192 frames x2 -> 384 CTC steps, all clips, 30 epochs")

GPU      : Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
Internet : ON
HF_TOKEN : found, 37 chars

full run armed: 192 frames x2 -> 384 CTC steps, all clips, 30 epochs


In [2]:
# =============================================================================
#  SanketVani (सङ्केतवाणी) — Continuous Sign Language Translation on iSign
#  CELL 1 / 2 : PARALLEL, ZERO-EXTRACTION HDF5 DATASET BUILDER
# =============================================================================
#  Streams the iSign MediaPipe-Holistic keypoint archive straight out of the
#  HuggingFace CDN with multi-threaded HTTP Range requests and packs it into a
#  SINGLE compressed HDF5 file.
#
#  Why no unzip:  the iSign keypoint archive holds 100k+ tiny `.pose` records.
#  Extracting them onto Kaggle's overlay filesystem exhausts the inode budget
#  long before the byte budget and the session dies with OSError 59 / "disk
#  quota exceeded".  Here nothing but the final .h5 ever touches the disk.
#
#  Self-contained: this file is one notebook cell.  Run it, then run Cell 2.
# =============================================================================

# ------------------------------------------------------------------ 0. CONFIG
CONFIG = {
    # --- source -----------------------------------------------------------
    "HF_REPO":        "Exploration-Lab/iSign",   # HuggingFace dataset repo id
    "HF_REVISION":    "main",
    # Leave this None.  The token is read from the HF_TOKEN environment
    # variable, then from the Kaggle secret named HF_TOKEN (Add-ons ->
    # Secrets), then from a cached `huggingface-cli login`.  Do NOT paste a
    # token here: notebooks get shared, exported and committed, and a token
    # in this cell travels with them.
    "HF_TOKEN":       None,

    # --- output -----------------------------------------------------------
    "OUT_H5":         "/kaggle/working/isign_ctc_dataset.h5",

    # --- tensor geometry (MUST stay in lock-step with the JS frontend) -----
    "NUM_FRAMES":     120,         # temporal length after uniform resampling
    "FEATURE_DIM":    225,         # 33*3 pose + 21*3 left hand + 21*3 right hand
    "STORE_DTYPE":    "float16",   # halves the .h5 footprint; lossless enough
                                   # for [0,1] landmarks (~5e-4 resolution)

    # --- ingestion --------------------------------------------------------
    "NUM_WORKERS":    12,          # ThreadPoolExecutor size for Range fetches
    "MAX_RETRIES":    5,           # exponential backoff on every HTTP call
    "BLOCK_SIZE":     1 << 20,     # 1 MiB aligned read-ahead blocks
    "BLOCK_CACHE":    8,           # blocks kept per thread (8 MiB / thread)
    "FULL_GET_LIMIT": 256 << 20,   # cap when an endpoint ignores Range headers
    "STREAM_BUFFER":  32 << 20,    # sequential read-ahead for tar containers
    "FLUSH_EVERY":    256,         # samples buffered before an HDF5 append
    "GZIP_LEVEL":     4,

    # --- preprocessing ----------------------------------------------------
    "TRIM_EMPTY":     True,        # drop leading/trailing untracked frames
    "MIN_FRAMES":     8,           # reject clips shorter than this
    "MIN_CHARS":      2,           # reject degenerate transcripts

    # --- optional manual overrides (leave None for full auto-discovery) ----
    "LABEL_FILE":     None,        # e.g. "iSign_v1.1.csv"
    "ID_COLUMN":      None,        # e.g. "uid"
    "TEXT_COLUMN":    None,        # e.g. "text"
    "ARCHIVE_GLOB":   None,        # e.g. "iSign_v1.1_keypoints.zip.part_*"

    # --- debug ------------------------------------------------------------
    "LIMIT_SAMPLES":  None,        # int -> build a small subset only
    "SUBSET_SEED":    1337,        # reproducible subset draw
    "SMOKE_TEST":     False,       # True -> synthesise a tiny fake .h5 instead
                                   #         (pipeline check only, NOT a model)
}

# --------------------------------------------------------- 1. DEPENDENCY BOOT
import importlib
import subprocess
import sys


def _ensure(module: str, pip_name: str = None, hard: bool = True):
    """Import `module`, pip-installing it once if it is missing."""
    try:
        return importlib.import_module(module)
    except ImportError:
        pass
    pkg = pip_name or module
    print(f"[boot] installing {pkg} ...", flush=True)
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--no-input", pkg],
            check=True, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
        )
        importlib.invalidate_caches()
        return importlib.import_module(module)
    except Exception as exc:                                   # noqa: BLE001
        if hard:
            raise RuntimeError(f"could not install required package '{pkg}'") from exc
        print(f"[boot] optional package '{pkg}' unavailable ({exc}); "
              f"falling back to the built-in implementation.", flush=True)
        return None


np       = _ensure("numpy")
h5py     = _ensure("h5py")
requests = _ensure("requests")
tqdm_mod = _ensure("tqdm", hard=False)
# pose-format is *preferred* but optional: a self-validating built-in parser
# takes over transparently when it cannot be installed (see PoseCodec below).
pose_fmt = _ensure("pose_format", "pose-format", hard=False)

tqdm = tqdm_mod.tqdm if tqdm_mod else (lambda x, **k: x)

import csv
import gc
import io
import json
import os
import posixpath
import re
import struct
import tarfile
import threading
import time
import zipfile
import zlib
from collections import OrderedDict, deque
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait

def _apply_env_overrides(config, prefix="SV_"):
    """Let a small paste-and-run cell override any CONFIG key via SV_<KEY>.

    Values are parsed as JSON when they can be (300, 0.5, true, null) and kept
    as a literal string otherwise, so SV_OUT_H5=/kaggle/working/x.h5 works too.
    Set a key back to its default with the string "none".
    """
    changed = []
    for key in list(config):
        raw = os.environ.get(prefix + key)
        if raw is None:
            continue
        text = raw.strip()
        if text.lower() in ("", "none", "null"):
            value = None
        else:
            try:
                value = json.loads(text)
            except ValueError:
                value = text
        config[key] = value
        changed.append(f"{key}=<hidden>" if "TOKEN" in key else f"{key}={value!r}")
    if changed:
        print("[config] overrides from environment: " + ", ".join(changed))
    return config


_apply_env_overrides(CONFIG)


# --------------------------------------------------------------- 2. HF ACCESS
HF_ENDPOINT = os.environ.get("HF_ENDPOINT", "https://huggingface.co").rstrip("/")


def _resolve_token(explicit=None):
    """-> (token, source).  env var -> Kaggle secret -> cached CLI login."""
    if explicit:
        return explicit.strip(), "CONFIG['HF_TOKEN']"
    for key in ("HF_TOKEN", "HUGGINGFACE_HUB_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        value = os.environ.get(key)
        if value and value.strip():
            return value.strip(), f"${key}"
    try:                                        # Kaggle "Add-ons -> Secrets"
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret("HF_TOKEN")
        if value and value.strip():
            return value.strip(), "Kaggle secret HF_TOKEN"
    except Exception:                                          # noqa: BLE001
        pass
    try:
        cached = os.path.expanduser("~/.cache/huggingface/token")
        if os.path.isfile(cached):
            value = open(cached).read().strip()
            if value:
                return value, "~/.cache/huggingface/token"
    except Exception:                                          # noqa: BLE001
        pass
    return None, None


TOKEN, TOKEN_SOURCE = _resolve_token(CONFIG["HF_TOKEN"])
AUTH_HEADERS = {"Authorization": f"Bearer {TOKEN}"} if TOKEN else {}
# The token itself is never printed or logged — only where it came from — so a
# shared notebook output cannot leak it.  Error messages carry URLs, not headers.
if TOKEN:
    print(f"[auth] HuggingFace token loaded from {TOKEN_SOURCE}"
          + ("" if TOKEN.startswith("hf_") else
             "  (warning: does not look like an 'hf_...' token — truncated paste?)"))
else:
    print("[auth] no HuggingFace token found — anonymous access. If iSign is "
          "gated for your account, add a Kaggle secret named HF_TOKEN "
          "(Add-ons -> Secrets) or export HF_TOKEN=... and re-run.")


def new_session() -> "requests.Session":
    """A Session with a wide connection pool (one per ingestion thread)."""
    s = requests.Session()
    adapter = requests.adapters.HTTPAdapter(
        pool_connections=CONFIG["NUM_WORKERS"] + 4,
        pool_maxsize=CONFIG["NUM_WORKERS"] + 4,
        max_retries=0,                       # we drive our own backoff below
    )
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    s.headers.update({"User-Agent": "SanketVani/1.0 (+iSign-CTC)"})
    s.headers.update(AUTH_HEADERS)
    return s


def http_with_backoff(session, method, url, *, headers=None, expect=(200,),
                      stream=False, timeout=60):
    """Any HTTP call, with exponential backoff (0.5s, 1s, 2s, 4s, 8s)."""
    last = None
    for attempt in range(CONFIG["MAX_RETRIES"]):
        try:
            r = session.request(method, url, headers=headers, stream=stream,
                                timeout=timeout, allow_redirects=True)
            if r.status_code in expect:
                return r
            # 4xx that will never succeed on retry -> fail fast.
            if r.status_code in (401, 403, 404, 416):
                r.close()
                raise RuntimeError(f"HTTP {r.status_code} for {url}")
            last = RuntimeError(f"HTTP {r.status_code} for {url}")
            r.close()
        except RuntimeError:
            raise
        except Exception as exc:                               # noqa: BLE001
            last = exc
        time.sleep(0.5 * (2 ** attempt))
    raise RuntimeError(f"request failed after {CONFIG['MAX_RETRIES']} attempts: {last}")


def hf_tree(repo, revision):
    """Recursive file listing of a HuggingFace dataset repo."""
    url = f"{HF_ENDPOINT}/api/datasets/{repo}/tree/{revision}?recursive=1&expand=1"
    entries = []
    with new_session() as s:
        while url:
            r = http_with_backoff(s, "GET", url)
            entries.extend(r.json())
            # large repos are paged; the cursor arrives in a Link header
            url = (r.links.get("next") or {}).get("url")
    out = []
    for e in entries:
        if e.get("type") != "file":
            continue
        size = e.get("size") or 0
        lfs = e.get("lfs") or {}
        size = lfs.get("size", size) or size
        out.append({"path": e["path"], "size": int(size)})
    return out


def resolve_url(repo, revision, path):
    return f"{HF_ENDPOINT}/datasets/{repo}/resolve/{revision}/{path}"


def head_size(session, url):
    r = http_with_backoff(session, "HEAD", url, expect=(200, 206))
    n = r.headers.get("x-linked-size") or r.headers.get("Content-Length")
    r.close()
    return int(n) if n else 0


# ------------------------------------------- 3. VIRTUAL CONCATENATED ARCHIVE
_NO_RANGE_WARNED = set()


def _warn_no_range(url):
    host = url.split("/")[2] if "://" in url else url
    if host not in _NO_RANGE_WARNED:
        _NO_RANGE_WARNED.add(host)
        print(f"[http] {host} ignores Range requests; slicing full responses "
              f"locally (slower, but correct).")


class RemoteConcatFile(io.RawIOBase):
    """A seekable, read-only file over N remote parts concatenated in order.

    `split -b` style multi-volume archives (`*.part_aa` .. `*.part_ad`) are a
    plain byte-wise split of one big file, so the logical archive is simply the
    concatenation of the parts.  Every read is served by an HTTP Range request
    against whichever part(s) the requested span falls into, with 1 MiB aligned
    read-ahead blocks so that walking the ZIP central directory is cheap.
    """

    def __init__(self, parts, session=None):
        self.parts = list(parts)                    # [(url, size), ...]
        self.session = session or new_session()
        self.offsets, acc = [], 0
        for _, size in self.parts:
            self.offsets.append(acc)
            acc += size
        self.size = acc
        self._pos = 0
        self._cache = OrderedDict()                 # block_index -> bytes

    # -- io.RawIOBase contract -------------------------------------------
    def readable(self):  return True
    def seekable(self):  return True
    def writable(self):  return False
    def tell(self):      return self._pos

    def seek(self, offset, whence=io.SEEK_SET):
        if whence == io.SEEK_SET:
            self._pos = offset
        elif whence == io.SEEK_CUR:
            self._pos += offset
        elif whence == io.SEEK_END:
            self._pos = self.size + offset
        else:
            raise ValueError(f"bad whence {whence}")
        self._pos = max(0, min(self._pos, self.size))
        return self._pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self._pos
        data = self.pread(self._pos, n)
        self._pos += len(data)
        return data

    def readinto(self, b):
        chunk = self.read(len(b))
        b[:len(chunk)] = chunk
        return len(chunk)

    # -- range machinery --------------------------------------------------
    def _fetch_span(self, start, length):
        """Raw bytes [start, start+length) across part boundaries."""
        out = bytearray()
        remaining = length
        cursor = start
        while remaining > 0 and cursor < self.size:
            idx = self._part_of(cursor)
            url, psize = self.parts[idx]
            base = self.offsets[idx]
            local = cursor - base
            take = min(remaining, psize - local)
            headers = {"Range": f"bytes={local}-{local + take - 1}"}
            r = http_with_backoff(self.session, "GET", url, headers=headers,
                                  expect=(206, 200), stream=True, timeout=120)
            try:
                if r.status_code == 206:
                    buf = r.content
                else:
                    # Some mirrors and corporate proxies silently drop the
                    # Range header and answer 200 with the whole volume.
                    # Slice it locally when that is affordable, otherwise say
                    # so plainly instead of pulling GiBs per read.
                    if psize > CONFIG["FULL_GET_LIMIT"]:
                        raise RuntimeError(
                            f"{url} ignored the Range header (HTTP 200) and the "
                            f"volume is {psize / 2**30:.1f} GiB — this endpoint "
                            f"cannot be streamed; use a Range-capable mirror.")
                    _warn_no_range(url)
                    buf = r.content[local:local + take]
            finally:
                r.close()
            if len(buf) != take:                    # short read -> retry once
                raise RuntimeError(
                    f"range short read: wanted {take} got {len(buf)} @ {url}")
            out += buf
            cursor += take
            remaining -= take
        return bytes(out)

    def _part_of(self, offset):
        lo, hi = 0, len(self.parts) - 1
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if self.offsets[mid] <= offset:
                lo = mid
            else:
                hi = mid - 1
        return lo

    def pread(self, offset, length):
        """Positional read; small reads go through the aligned block cache."""
        if offset >= self.size or length <= 0:
            return b""
        length = min(length, self.size - offset)
        bs = CONFIG["BLOCK_SIZE"]
        if length > bs:                             # big contiguous read
            return self._fetch_span(offset, length)
        out = bytearray()
        pos = offset
        end = offset + length
        while pos < end:
            bi = pos // bs
            block = self._cache.get(bi)
            if block is None:
                bstart = bi * bs
                block = self._fetch_span(bstart, min(bs, self.size - bstart))
                self._cache[bi] = block
                while len(self._cache) > CONFIG["BLOCK_CACHE"]:
                    self._cache.popitem(last=False)
            else:
                self._cache.move_to_end(bi)
            inner = pos - bi * bs
            take = min(len(block) - inner, end - pos)
            if take <= 0:
                break
            out += block[inner:inner + take]
            pos += take
        return bytes(out)

    def close(self):
        try:
            self.session.close()
        except Exception:                                      # noqa: BLE001
            pass
        self._cache.clear()
        super().close()


class ZipIndex:
    """ZIP central directory read once; members extracted per-thread after."""

    def __init__(self, parts):
        reader = RemoteConcatFile(parts)
        try:
            with zipfile.ZipFile(reader) as zf:
                self.entries = [
                    {
                        "name": zi.filename,
                        "offset": zi.header_offset,
                        "csize": zi.compress_size,
                        "usize": zi.file_size,
                        "method": zi.compress_type,
                        "encrypted": bool(zi.flag_bits & 0x1),
                        "namelen": len(zi.orig_filename.encode("utf-8", "replace")),
                    }
                    for zi in zf.infolist() if not zi.is_dir()
                ]
        finally:
            reader.close()

    @staticmethod
    def read_member(reader: RemoteConcatFile, ent) -> bytes:
        """Local header + payload in ONE range request, inflated in memory."""
        if ent["encrypted"]:
            raise ValueError("encrypted zip member")
        guess = 30 + ent["namelen"] + 256
        blob = reader.pread(ent["offset"], guess + ent["csize"])
        if len(blob) < 30 or blob[:4] != b"PK\x03\x04":
            raise ValueError("bad local file header signature")
        n, m = struct.unpack("<HH", blob[26:30])
        head = 30 + n + m
        if head + ent["csize"] > len(blob):          # extra field bigger than
            blob = reader.pread(ent["offset"],       # our guess -> exact refetch
                                head + ent["csize"])
        raw = blob[head:head + ent["csize"]]
        if len(raw) != ent["csize"]:
            raise ValueError("truncated member payload")
        if ent["method"] == zipfile.ZIP_STORED:
            return raw
        if ent["method"] == zipfile.ZIP_DEFLATED:
            return zlib.decompress(raw, -zlib.MAX_WBITS)
        if ent["method"] == zipfile.ZIP_BZIP2:
            import bz2
            return bz2.decompress(raw)
        raise ValueError(f"unsupported zip compression method {ent['method']}")


# ------------------------------------------------------------ 4. POSE DECODER
POSE_N, HAND_N = 33, 21
SLOT_POSE  = (0,   99)     # 33 landmarks * (x, y, z)
SLOT_LEFT  = (99,  162)    # 21 landmarks * (x, y, z)
SLOT_RIGHT = (162, 225)    # 21 landmarks * (x, y, z)


class PoseCodec:
    """Decodes `.pose` (sign-language-processing/pose) blobs from memory.

    Primary path uses the `pose-format` library.  If it is unavailable the
    built-in reader takes over: it parses the header, then *proves* its body
    layout by checking that the declared tensor shape consumes the remaining
    bytes exactly.  Candidate layouts are tried in order and a blob that
    matches none is reported as corrupt rather than silently mis-parsed.
    """

    def __init__(self):
        self.use_lib = pose_fmt is not None
        self._Pose = None
        if self.use_lib:
            try:
                from pose_format import Pose
                self._Pose = Pose
            except Exception:                                  # noqa: BLE001
                self.use_lib = False

    # -- public ------------------------------------------------------------
    def decode(self, blob: bytes):
        """-> (data[frames, points, 3] float32, comps[(name, n_points)], (w, h))"""
        if self.use_lib:
            try:
                return self._decode_lib(blob)
            except Exception:                                  # noqa: BLE001
                pass                                # fall through to built-in
        return self._decode_builtin(blob)

    # -- library path ------------------------------------------------------
    def _decode_lib(self, blob):
        pose = self._Pose.read(blob)
        arr = pose.body.data                        # (frames, people, pts, dims)
        conf = pose.body.confidence
        arr = np.ma.filled(arr, 0.0).astype(np.float32)
        conf = np.ma.filled(conf, 0.0).astype(np.float32)
        if arr.ndim != 4 or arr.shape[0] == 0:
            raise ValueError("empty pose body")
        arr = arr[:, 0]                             # first (only) person
        conf = conf[:, 0]
        arr = np.where(conf[..., None] > 0, arr, 0.0)
        arr = self._to_xyz(arr)
        comps = [(c.name, len(c.points)) for c in pose.header.components]
        dims = pose.header.dimensions
        return arr, comps, (float(dims.width), float(dims.height))

    # -- built-in path -----------------------------------------------------
    def _decode_builtin(self, blob):
        mv = memoryview(blob)
        off = 0

        def u16():
            nonlocal off
            v = struct.unpack_from("<H", mv, off)[0]; off += 2; return v

        def s():
            nonlocal off
            ln = u16()
            v = bytes(mv[off:off + ln]).decode("utf-8", "replace"); off += ln
            return v

        if len(blob) < 16:
            raise ValueError("pose blob too small")
        version = struct.unpack_from("<f", mv, off)[0]; off += 4
        width, height, depth = u16(), u16(), u16()
        ncomp = u16()
        if not (0 < ncomp <= 64):
            raise ValueError(f"implausible component count {ncomp}")
        comps, total_points, max_dims = [], 0, 0
        for _ in range(ncomp):
            name = s()
            fmt = s()
            npts, nlimbs, ncols = u16(), u16(), u16()
            for _ in range(npts):
                s()
            off += nlimbs * 4
            off += ncols * 6
            comps.append((name, npts))
            total_points += npts
            max_dims = max(max_dims, len(fmt) - 1)   # "XYZC" -> 3
        if total_points == 0 or max_dims <= 0:
            raise ValueError("degenerate pose header")

        body = off
        rest = len(blob) - body
        for frame_fmt in ("<fHH", "<fIH", "<fIHHH"):
            try:
                fields = struct.unpack_from(frame_fmt, mv, body)
            except struct.error:
                continue
            hdr_len = struct.calcsize(frame_fmt)
            frames, people = int(fields[1]), int(fields[2])
            pts, dims = total_points, max_dims
            if frame_fmt == "<fIHHH":
                pts, dims = int(fields[3]), int(fields[4])
            if frames <= 0 or people <= 0 or pts <= 0 or dims <= 0:
                continue
            need = frames * people * pts * (dims + 1) * 4      # data + conf
            if rest - hdr_len != need:
                continue
            start = body + hdr_len
            ndata = frames * people * pts * dims
            data = np.frombuffer(blob, np.float32, ndata, start)
            conf = np.frombuffer(blob, np.float32, frames * people * pts,
                                 start + ndata * 4)
            data = data.reshape(frames, people, pts, dims)[:, 0]
            conf = conf.reshape(frames, people, pts)[:, 0]
            data = np.where(conf[..., None] > 0, data, 0.0).astype(np.float32)
            return self._to_xyz(data), comps, (float(width), float(height))
        raise ValueError(f"unrecognised .pose body layout (v{version:.2f})")

    @staticmethod
    def _to_xyz(arr):
        """Force the last axis to exactly (x, y, z), zero-padding a missing z."""
        d = arr.shape[-1]
        if d == 3:
            return arr
        if d > 3:
            return np.ascontiguousarray(arr[..., :3])
        out = np.zeros(arr.shape[:-1] + (3,), np.float32)
        out[..., :d] = arr
        return out


CODEC = PoseCodec()


def component_slices(comps):
    """Locate pose / left-hand / right-hand blocks inside the point axis.

    Primary: MediaPipe Holistic component names.  Fallback: geometry — the one
    33-point component is the body, and the two 21-point components are the
    hands in left-then-right file order (pose-format's canonical ordering).
    """
    bounds, acc = [], 0
    for name, n in comps:
        bounds.append((name.lower(), acc, acc + n, n))
        acc += n
    pose = left = right = None
    for name, a, b, n in bounds:
        flat = name.replace("-", "_").replace(" ", "_")
        if pose is None and n == POSE_N and "pose" in flat:
            pose = (a, b)
        elif left is None and n == HAND_N and "left" in flat:
            left = (a, b)
        elif right is None and n == HAND_N and "right" in flat:
            right = (a, b)
    if pose is None:
        cands = [(a, b) for _, a, b, n in bounds if n == POSE_N]
        pose = cands[0] if cands else None
    if left is None or right is None:
        hands = [(a, b) for _, a, b, n in bounds if n == HAND_N]
        if len(hands) >= 2:
            left = left or hands[0]
            right = right or hands[1]
        elif len(hands) == 1:
            left = left or hands[0]
    if pose is None:
        raise ValueError("no 33-point body component in pose header")
    return pose, left, right


def features_from_pose(blob):
    """`.pose` bytes -> (NUM_FRAMES, 225) float32 in MediaPipe-normalised units."""
    data, comps, (width, height) = CODEC.decode(blob)
    p_sl, l_sl, r_sl = component_slices(comps)
    F, T = CONFIG["FEATURE_DIM"], CONFIG["NUM_FRAMES"]

    n_raw = data.shape[0]
    feats = np.zeros((n_raw, F), np.float32)

    def put(sl, slot, expect):
        if sl is None:
            return                                  # untracked -> stays 0.0
        blk = data[:, sl[0]:sl[1], :]
        if blk.shape[1] != expect:                  # defensive crop / pad
            fixed = np.zeros((n_raw, expect, 3), np.float32)
            k = min(expect, blk.shape[1])
            fixed[:, :k] = blk[:, :k]
            blk = fixed
        feats[:, slot[0]:slot[1]] = blk.reshape(n_raw, -1)

    put(p_sl, SLOT_POSE, POSE_N)
    put(l_sl, SLOT_LEFT, HAND_N)
    put(r_sl, SLOT_RIGHT, HAND_N)

    # --- scale restoration -------------------------------------------------
    # pose-format stores landmarks multiplied by the source video dimensions.
    # MediaPipe Holistic JS emits x, y in [0, 1] with z on the same scale as x,
    # so we divide by (w, h, w) to land the training distribution exactly on
    # the frontend's.  Archives already written in normalised units declare
    # width <= 1 and are passed through untouched — no ad-hoc shifts anywhere.
    if width > 1.0 and height > 1.0:
        scale = np.tile(np.array([1.0 / width, 1.0 / height, 1.0 / width],
                                 np.float32), F // 3)
        feats *= scale

    # --- hygiene -----------------------------------------------------------
    np.nan_to_num(feats, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

    if CONFIG["TRIM_EMPTY"]:
        active = np.abs(feats[:, SLOT_POSE[0]:SLOT_POSE[1]]).sum(1) > 0
        if active.any():
            lo, hi = int(np.argmax(active)), int(len(active) - np.argmax(active[::-1]))
            feats = feats[lo:hi]

    if feats.shape[0] < CONFIG["MIN_FRAMES"]:
        raise ValueError(f"only {feats.shape[0]} tracked frames")

    # --- uniform temporal resampling to a fixed T --------------------------
    # Identical to the JS frontend: sample T evenly spaced source indices
    # (duplicating frames when the clip is shorter than T).  Never truncates,
    # so the tail of a sentence is preserved.
    # floor(x + 0.5), not np.rint: numpy rounds halves to even while JS
    # Math.round rounds them up, and that would pick a different source frame.
    idx = np.linspace(0, feats.shape[0] - 1, T)
    idx = np.floor(idx + 0.5).astype(np.int64)
    return np.ascontiguousarray(feats[idx]), n_raw


# ---------------------------------------------------------------- 5. LABELS
ID_HINTS = ("uid", "id", "file", "filename", "file_name", "video", "video_id",
            "pose", "pose_file", "name", "key", "vid", "clip", "sample")
TEXT_HINTS = ("text", "translation", "sentence", "english", "label", "caption",
              "transcript", "transcription", "target", "gloss", "annotation")
DROP_EXT = (".pose", ".mp4", ".mov", ".avi", ".webm", ".mkv", ".json", ".npy")


def norm_key(value: str) -> str:
    """Canonical join key: lowercase basename with any media extension gone."""
    v = str(value).strip().replace("\\", "/")
    v = posixpath.basename(v).lower()
    for _ in range(2):
        stem, ext = posixpath.splitext(v)
        if ext in DROP_EXT:
            v = stem
        else:
            break
    return v


def _rows_from_bytes(path, raw):
    """Best-effort table reader for csv / tsv / json / jsonl / parquet bytes."""
    low = path.lower()
    try:
        if low.endswith(".parquet"):
            import pandas as pd
            return pd.read_parquet(io.BytesIO(raw)).to_dict("records")
        text = raw.decode("utf-8", "replace")
        if low.endswith(".jsonl") or low.endswith(".ndjson"):
            return [json.loads(l) for l in text.splitlines() if l.strip()]
        if low.endswith(".json"):
            obj = json.loads(text)
            if isinstance(obj, dict):
                # {"split": [...]} or {"uid": {...}} shapes
                for v in obj.values():
                    if isinstance(v, list) and v and isinstance(v[0], dict):
                        return v
                return [dict(v, __key__=k) for k, v in obj.items()
                        if isinstance(v, dict)]
            return [o for o in obj if isinstance(o, dict)]
        delim = "\t" if low.endswith((".tsv", ".tab")) else ","
        return list(csv.DictReader(io.StringIO(text), delimiter=delim))
    except Exception as exc:                                   # noqa: BLE001
        print(f"[labels] skipping {path}: {exc}")
        return []


def _pick(columns, hints, override):
    if override and override in columns:
        return override
    cols = {c.lower(): c for c in columns if c}
    for h in hints:                                # exact name wins
        if h in cols:
            return cols[h]
    for h in hints:                                # then substring
        for lc, orig in cols.items():
            if h in lc:
                return orig
    return None


def discover_labels(tree):
    """Auto-locate and parse the transcript table -> {norm_key: text}."""
    cands = [f for f in tree
             if f["path"].lower().endswith((".csv", ".tsv", ".json", ".jsonl",
                                            ".ndjson", ".parquet"))
             and not posixpath.basename(f["path"]).startswith(".")
             and f["path"] not in ("dataset_infos.json",)
             and f["size"] < 400 * 1024 * 1024]
    if CONFIG["LABEL_FILE"]:
        cands = [f for f in cands if CONFIG["LABEL_FILE"] in f["path"]]
    cands.sort(key=lambda f: -f["size"])            # richest table first
    if not cands:
        raise RuntimeError("no transcript table (.csv/.json/.parquet) in the repo")

    mapping, report, budget = {}, [], 1 << 30      # at most 1 GiB of tables
    with new_session() as s:
        for f in cands[:12]:
            if budget <= 0:
                break
            budget -= f["size"]
            url = resolve_url(CONFIG["HF_REPO"], CONFIG["HF_REVISION"], f["path"])
            try:
                raw = http_with_backoff(s, "GET", url, timeout=180).content
            except Exception as exc:                           # noqa: BLE001
                print(f"[labels] cannot download {f['path']}: {exc}")
                continue
            rows = _rows_from_bytes(f["path"], raw)
            if not rows:
                continue
            cols = list(rows[0].keys())
            id_col = _pick(cols, ID_HINTS, CONFIG["ID_COLUMN"])
            tx_col = _pick(cols, TEXT_HINTS, CONFIG["TEXT_COLUMN"])
            report.append(f"    {f['path']}  columns={cols}")
            if not id_col or not tx_col or id_col == tx_col:
                continue
            added = 0
            for row in rows:
                key, txt = row.get(id_col), row.get(tx_col)
                if key is None or not isinstance(txt, str) or not txt.strip():
                    continue
                nk = norm_key(key)
                if nk and nk not in mapping:
                    mapping[nk] = txt.strip()
                    added += 1
            print(f"[labels] {f['path']}: +{added} pairs "
                  f"(id='{id_col}', text='{tx_col}')")
    if not mapping:
        raise RuntimeError(
            "could not auto-detect the id/text columns.  Inspect the tables "
            "below and set CONFIG['LABEL_FILE'/'ID_COLUMN'/'TEXT_COLUMN']:\n"
            + "\n".join(report))
    return mapping


# --------------------------------------------------------- 6. ARCHIVE LOOKUP
# Volume suffixes appear as `name.zip.part_aa`, `name_part_aa` (iSign),
# `name.001` or `name.z01`, so the separator before the suffix may be any of
# `. _ -` — not a dot alone.
SPLIT_RE = re.compile(
    r"^(?P<stem>.+?)[._-](?:part[._-]?|z|)(?P<idx>[a-z]{2}|\d{2,3})$",
    re.IGNORECASE)


def find_archive_parts(tree):
    """Group `*.part_aa .. *.part_ad` (or .001/.z01) into one ordered archive."""
    if CONFIG["ARCHIVE_GLOB"]:
        import fnmatch
        sel = sorted((f for f in tree
                      if fnmatch.fnmatch(f["path"], CONFIG["ARCHIVE_GLOB"])),
                     key=lambda f: f["path"])
        if sel:
            return sel
    groups = {}
    for f in tree:
        m = SPLIT_RE.match(f["path"])
        if not m:
            continue
        stem = m.group("stem").lower()
        if (".zip" in stem or stem.endswith((".tar", ".tgz", ".gz", ".7z"))
                or "part" in f["path"].lower() or m.group("idx").isdigit()):
            groups.setdefault(m.group("stem"), []).append(f)
    if groups:
        # keypoints beat video, and among equals the biggest set wins
        def score(item):
            stem, files = item
            low = stem.lower()
            keypoints = any(k in low for k in
                            ("keypoint", "pose", "landmark", "skeleton"))
            video = any(k in low for k in ("video", "rgb", "mp4", "frame"))
            return (keypoints, not video, sum(f["size"] for f in files))

        for stem, files in sorted(groups.items()):
            print(f"[archive] candidate '{stem}' -> {len(files)} volumes, "
                  f"{sum(f['size'] for f in files) / 2**30:.1f} GiB")
        stem, files = max(groups.items(), key=score)
        files.sort(key=lambda f: f["path"])
        print(f"[archive] selected '{stem}' -> {len(files)} volumes")
        return files
    singles = [f for f in tree if f["path"].lower().endswith(".zip")]
    if singles:
        best = max(singles, key=lambda f: f["size"])
        print(f"[archive] single archive '{best['path']}'")
        return [best]
    raise RuntimeError("no keypoint archive found in the repo tree")


# ------------------------------------------------------------- 7. HDF5 BUILD
def _open_h5(path, n_hint):
    T, F = CONFIG["NUM_FRAMES"], CONFIG["FEATURE_DIM"]
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    h5 = h5py.File(path, "w", libver="latest", rdcc_nbytes=64 * 1024 * 1024)
    str_dt = h5py.string_dtype("utf-8")
    ds = {
        "features": h5.create_dataset(
            "features", shape=(0, T, F), maxshape=(None, T, F),
            dtype=CONFIG["STORE_DTYPE"], chunks=(1, T, F),
            compression="gzip", compression_opts=CONFIG["GZIP_LEVEL"],
            shuffle=True),
        "text": h5.create_dataset("text", shape=(0,), maxshape=(None,),
                                  dtype=str_dt, chunks=(512,),
                                  compression="gzip"),
        "name": h5.create_dataset("name", shape=(0,), maxshape=(None,),
                                  dtype=str_dt, chunks=(512,),
                                  compression="gzip"),
        "n_frames_raw": h5.create_dataset("n_frames_raw", shape=(0,),
                                          maxshape=(None,), dtype="int32",
                                          chunks=(512,), compression="gzip"),
    }
    h5.attrs.update({
        "dataset": CONFIG["HF_REPO"], "revision": CONFIG["HF_REVISION"],
        "num_frames": T, "feature_dim": F,
        "layout": "pose[0:99] left_hand[99:162] right_hand[162:225]; xyz per landmark",
        "coord_space": "mediapipe_normalized(x,y in [0,1]; z ~ x scale)",
        "resample": "uniform_nearest_to_120", "complete": False,
        "expected": int(n_hint),
    })
    return h5, ds


def _append(ds, feats, texts, names, raws):
    n = len(texts)
    if not n:
        return
    for key, values in (("features", feats), ("text", texts),
                        ("name", names), ("n_frames_raw", raws)):
        d = ds[key]
        cur = d.shape[0]
        d.resize(cur + n, axis=0)
        d[cur:cur + n] = (np.asarray(values, dtype=CONFIG["STORE_DTYPE"])
                          if key == "features" else values)


def build_smoke_dataset(path, n=192):
    """Deterministic synthetic .h5 so Cell 2 can be validated without network."""
    print("[smoke] building SYNTHETIC dataset — for pipeline checks only.")
    rng = np.random.default_rng(0)
    words = ["hello", "how are you", "my name is ravi", "thank you",
             "what is your name", "i am going home", "please help me"]
    h5, ds = _open_h5(path, n)
    T, F = CONFIG["NUM_FRAMES"], CONFIG["FEATURE_DIM"]
    feats, texts, names, raws = [], [], [], []
    for i in range(n):
        txt = words[i % len(words)]
        base = rng.random((1, F), dtype=np.float32) * 0.6 + 0.2
        wave = np.sin(np.linspace(0, 3 * np.pi * (1 + (i % 5)), T))[:, None]
        x = (base + 0.05 * wave * (hash(txt) % 7 + 1) / 7).astype(np.float32)
        x[:, SLOT_RIGHT[0]:SLOT_RIGHT[1]] *= (i % 2)     # a missing-hand case
        feats.append(x); texts.append(txt); names.append(f"smoke_{i:05d}")
        raws.append(T)
    _append(ds, feats, texts, names, raws)
    h5.attrs["complete"] = True
    h5.attrs["synthetic"] = True
    h5.close()
    print(f"[smoke] wrote {n} synthetic samples -> {path}")


# --------------------------------------------------- 7b. CONTAINER DETECTION
CONTAINER_MAGIC = (
    (b"PK\x03\x04", "zip"), (b"PK\x05\x06", "zip"), (b"PK\x07\x08", "zip"),
    (b"\x1f\x8b", "gzip"), (b"BZh", "bzip2"), (b"\xfd7zXZ\x00", "xz"),
    (b"7z\xbc\xaf\x27\x1c", "7z"), (b"\x28\xb5\x2f\xfd", "zstd"),
    (b"Rar!\x1a\x07", "rar"), (b"\x04\x22\x4d\x18", "lz4"),
)


def sniff_container(reader):
    """Identify the archive from its leading bytes.

    The iSign volumes carry no file extension at all, so the container has to
    be detected rather than inferred from the name.
    """
    head = reader.pread(0, 1024)
    for magic, kind in CONTAINER_MAGIC:
        if head.startswith(magic):
            return kind
    if len(head) >= 262 and head[257:262] == b"ustar":
        return "tar"
    raise RuntimeError(
        f"unrecognised archive container. First 16 bytes: {head[:16]!r} "
        f"(hex {head[:16].hex()}). Supported: zip, tar, tar.gz, tar.bz2, tar.xz.")


def iter_tar_members(reader, kind):
    """Sequential (name, bytes) over a tar, de-compressing on the fly.

    A compressed tar cannot be seeked into, so this is one ordered pass with a
    large read-ahead buffer: every underlying read is a single big Range
    request rather than thousands of small ones.
    """
    buffered = io.BufferedReader(reader, buffer_size=CONFIG["STREAM_BUFFER"])
    mode = {"tar": "r|", "gzip": "r|gz", "bzip2": "r|bz2", "xz": "r|xz"}[kind]
    with tarfile.open(fileobj=buffered, mode=mode) as tar:
        for member in tar:
            if not member.isfile():
                continue
            handle = tar.extractfile(member)
            if handle is not None:
                yield member.name, handle.read()


# ------------------------------------------------------------- 7c. PRODUCERS
def _decode_one(name, blob, text):
    """Corrupt / truncated / empty records return an error instead of raising."""
    try:
        feats, n_raw = features_from_pose(blob)
        return feats, text, name, n_raw, None
    except Exception as exc:                                   # noqa: BLE001
        return None, None, name, 0, f"{type(exc).__name__}: {exc}"


def _match_error(sample_names, labels):
    return ("transcript keys do not match .pose filenames.\n"
            f"  pose keys : {[norm_key(n) for n in sample_names]}\n"
            f"  label keys: {list(labels)[:5]}\n"
            "Set CONFIG['ID_COLUMN'] (SV_ID_COLUMN) to the column holding the "
            "pose filename.")


def _zip_producer(resolved, labels, limit):
    """Random access: the central directory is read once, then NUM_WORKERS
    threads each pull one member with a single Range request."""
    print("[zip] reading central directory over HTTP range ...")
    index = ZipIndex(resolved)
    members = [e for e in index.entries if e["name"].lower().endswith(".pose")]
    print(f"[zip] {len(index.entries)} members, {len(members)} .pose records")
    if not members:
        raise RuntimeError("archive contains no .pose members")

    work = [(ent, labels[norm_key(ent["name"])]) for ent in members
            if norm_key(ent["name"]) in labels]
    print(f"[match] {len(work)}/{len(members)} .pose records matched a transcript")
    if not work:
        raise RuntimeError(_match_error([m["name"] for m in members[:5]], labels))
    if limit and limit < len(work):
        # Draw the subset across the whole corpus.  Central-directory order
        # clusters by signer and topic, so the first N records are a far
        # narrower slice of the language than a random N.
        rng = np.random.default_rng(CONFIG["SUBSET_SEED"])
        picked = np.sort(rng.choice(len(work), size=limit, replace=False))
        work = [work[i] for i in picked]     # sorted: keeps read locality

    tls = threading.local()

    def job(item):
        ent, text = item
        reader = getattr(tls, "reader", None)
        if reader is None:
            reader = tls.reader = RemoteConcatFile(resolved)
        try:
            blob = ZipIndex.read_member(reader, ent)
        except Exception as exc:                               # noqa: BLE001
            return None, None, ent["name"], 0, f"{type(exc).__name__}: {exc}"
        return _decode_one(ent["name"], blob, text)

    def gen():
        pool = ThreadPoolExecutor(max_workers=CONFIG["NUM_WORKERS"])
        pending, queue = set(), iter(work)
        try:
            for _ in range(CONFIG["NUM_WORKERS"] * 3):   # bounded in-flight
                nxt = next(queue, None)
                if nxt is None:
                    break
                pending.add(pool.submit(job, nxt))
            while pending:
                done, pending = wait(pending, return_when=FIRST_COMPLETED)
                for fut in done:
                    yield fut.result()
                    nxt = next(queue, None)
                    if nxt is not None:
                        pending.add(pool.submit(job, nxt))
        finally:
            pool.shutdown(wait=True)

    return len(work), gen()


def _tar_producer(resolved, kind, labels, limit):
    """One ordered pass over the stream; decoding is overlapped with the
    download by a small pool, since the network is the bottleneck."""
    print(f"[tar] streaming a {kind} container in one pass "
          f"(no random access, so members arrive in archive order)")

    def gen():
        pool = ThreadPoolExecutor(max_workers=max(2, CONFIG["NUM_WORKERS"] // 3))
        window = max(8, CONFIG["NUM_WORKERS"] * 4)
        pending, seen, matched, first = deque(), 0, 0, []
        try:
            for name, blob in iter_tar_members(RemoteConcatFile(resolved), kind):
                if not name.lower().endswith(".pose"):
                    continue
                seen += 1
                if len(first) < 5:
                    first.append(name)
                text = labels.get(norm_key(name))
                if text is None:
                    # Fail fast rather than streaming 150 GiB for nothing.
                    if seen >= 2000 and matched == 0:
                        raise RuntimeError(_match_error(first, labels))
                    continue
                matched += 1
                # A sequential container cannot be sampled without reading it
                # all, so a limited tar run is necessarily a prefix.
                pending.append(pool.submit(_decode_one, name, blob, text))
                while len(pending) >= window:
                    yield pending.popleft().result()
                if limit and matched >= limit:
                    break
            while pending:
                yield pending.popleft().result()
        finally:
            pool.shutdown(wait=True)
            print(f"\n[match] {matched}/{seen} .pose records matched a transcript")

    return None, gen()


# -------------------------------------------------------------- 7d. CONSUMER
def _consume(results, total, out):
    """Single-threaded HDF5 writer — h5py handles are not thread-safe."""
    h5, ds = _open_h5(out, total or 0)
    buf_f, buf_t, buf_n, buf_r = [], [], [], []
    kept = skipped = 0
    reasons = {}
    bar = tqdm(total=total, desc="packing", unit="clip")
    try:
        for feats, text, name, n_raw, err in results:
            if err is None:
                buf_f.append(feats); buf_t.append(text)
                buf_n.append(name); buf_r.append(n_raw)
                kept += 1
            else:
                skipped += 1
                tag = err.split(":")[0]
                reasons[tag] = reasons.get(tag, 0) + 1
            if hasattr(bar, "update"):
                bar.update(1)
            if len(buf_f) >= CONFIG["FLUSH_EVERY"]:
                _append(ds, buf_f, buf_t, buf_n, buf_r)
                buf_f, buf_t, buf_n, buf_r = [], [], [], []
                del feats
        _append(ds, buf_f, buf_t, buf_n, buf_r)
    finally:
        if hasattr(bar, "close"):
            bar.close()
        h5.attrs["complete"] = kept > 0
        h5.attrs["n_samples"] = kept
        h5.attrs["n_skipped"] = skipped
        h5.close()
    return kept, skipped, reasons


def build_dataset():
    out = CONFIG["OUT_H5"]
    if CONFIG["SMOKE_TEST"]:
        build_smoke_dataset(out)
        return

    print(f"[hf] listing {CONFIG['HF_REPO']} @ {CONFIG['HF_REVISION']} ...")
    tree = hf_tree(CONFIG["HF_REPO"], CONFIG["HF_REVISION"])
    print(f"[hf] {len(tree)} files, "
          f"{sum(f['size'] for f in tree) / 2**30:.1f} GiB total")

    labels = discover_labels(tree)
    print(f"[labels] {len(labels)} id->text pairs")

    parts = find_archive_parts(tree)
    resolved = []
    with new_session() as s:
        for f in parts:
            url = resolve_url(CONFIG["HF_REPO"], CONFIG["HF_REVISION"], f["path"])
            size = f["size"] or head_size(s, url)
            if size <= 0:
                raise RuntimeError(f"cannot determine size of {f['path']}")
            resolved.append((url, size))
            print(f"[archive]   {f['path']}  {size / 2**30:.2f} GiB")
    total_bytes = sum(sz for _, sz in resolved)
    print(f"[archive] virtual archive = {total_bytes / 2**30:.2f} GiB "
          f"(nothing is written to disk)")

    probe = RemoteConcatFile(resolved)
    try:
        kind = sniff_container(probe)
    finally:
        probe.close()
    print(f"[archive] container detected: {kind}")

    limit = int(CONFIG["LIMIT_SAMPLES"]) if CONFIG["LIMIT_SAMPLES"] else None
    if kind == "zip":
        total, results = _zip_producer(resolved, labels, limit)
    elif kind in ("tar", "gzip", "bzip2", "xz"):
        if kind != "tar" and not limit:
            print(f"[tar] a {kind} stream cannot be seeked, so the full "
                  f"{total_bytes / 2**30:.0f} GiB will be downloaded once. "
                  f"Set SV_LIMIT_SAMPLES to stop early.")
        total, results = _tar_producer(resolved, kind, labels, limit)
    else:
        raise RuntimeError(
            f"container '{kind}' is not supported for streaming ingestion. "
            f"zip, tar, tar.gz, tar.bz2 and tar.xz are.")

    kept, skipped, reasons = _consume(results, total, out)

    print(f"\n[done] kept {kept}  |  skipped {skipped}  ->  {out}")
    if reasons:
        print(f"[done] skip reasons: {reasons}")
    print(f"[done] file size: {os.path.getsize(out) / 2**20:.1f} MiB")
    if kept == 0:
        raise RuntimeError("every record failed to decode — see skip reasons above")

    # ---- active memory management -----------------------------------------
    del results, labels, tree, parts
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:                                          # noqa: BLE001
        pass


def verify_dataset(path):
    with h5py.File(path, "r") as h5:
        n = h5["features"].shape[0]
        assert n > 0, "dataset is empty"
        x = np.asarray(h5["features"][0], np.float32)
        assert x.shape == (CONFIG["NUM_FRAMES"], CONFIG["FEATURE_DIM"]), x.shape
        assert np.isfinite(x).all(), "non-finite values leaked into the dataset"
        print(f"[verify] {n} samples | sample0 {x.shape} "
              f"| x-range [{x.min():.3f}, {x.max():.3f}] "
              f'| text: "{h5["text"][0].decode() if isinstance(h5["text"][0], bytes) else h5["text"][0]}"')
        print(f"[verify] attrs: {dict(h5.attrs)}")

t0 = time.time()
if os.path.exists(CONFIG["OUT_H5"]):
    try:
        with h5py.File(CONFIG["OUT_H5"], "r") as _h:
            done = bool(_h.attrs.get("complete", False))
    except Exception:                                      # noqa: BLE001
        done = False
    if done:
        print(f"[skip] {CONFIG['OUT_H5']} already complete — reusing it.")
    else:
        print("[rebuild] previous run left an incomplete file; rebuilding.")
        os.remove(CONFIG["OUT_H5"])
        build_dataset()
else:
    build_dataset()
verify_dataset(CONFIG["OUT_H5"])
print(f"[cell1] finished in {time.time() - t0:.1f}s")


[boot] installing pose-format ...
[config] overrides from environment: NUM_FRAMES=192, LIMIT_SAMPLES=None
[auth] HuggingFace token loaded from $HF_TOKEN
[hf] listing Exploration-Lab/iSign @ main ...
[hf] 11 files, 212.5 GiB total
[labels] iSign_v1.1.csv: +126846 pairs (id='uid', text='text')
[labels] word-presence-dataset_v1.1.csv: +0 pairs (id='word_id', text='sentence')
[labels] word-description-dataset_v1.1.csv: +0 pairs (id='word_id', text='sentence')
[labels] 126846 id->text pairs
[archive] candidate 'iSign-poses_v1.1' -> 4 volumes, 158.5 GiB
[archive] candidate 'iSign-videos_v1.1' -> 2 volumes, 53.9 GiB
[archive] selected 'iSign-poses_v1.1' -> 4 volumes
[archive]   iSign-poses_v1.1_part_aa  45.00 GiB
[archive]   iSign-poses_v1.1_part_ab  45.00 GiB
[archive]   iSign-poses_v1.1_part_ac  45.00 GiB
[archive]   iSign-poses_v1.1_part_ad  23.54 GiB
[archive] virtual archive = 158.54 GiB (nothing is written to disk)
[archive] container detected: zip
[zip] reading central directory over H

packing: 100%|██████████| 126846/126846 [2:34:16<00:00, 13.70clip/s]



[done] kept 123119  |  skipped 3727  ->  /kaggle/working/isign_ctc_dataset.h5
[done] skip reasons: {'ValueError': 30, 'RuntimeError': 3697}
[done] file size: 5351.5 MiB
[verify] 123119 samples | sample0 (192, 225) | x-range [-0.009, 2.166] | text: "Christians and Muslims bury the dead."
[verify] attrs: {'complete': np.True_, 'coord_space': 'mediapipe_normalized(x,y in [0,1]; z ~ x scale)', 'dataset': 'Exploration-Lab/iSign', 'expected': np.int64(126846), 'feature_dim': np.int64(225), 'layout': 'pose[0:99] left_hand[99:162] right_hand[162:225]; xyz per landmark', 'n_samples': np.int64(123119), 'n_skipped': np.int64(3727), 'num_frames': np.int64(192), 'resample': 'uniform_nearest_to_120', 'revision': 'main'}
[cell1] finished in 9268.5s


In [3]:
# =============================================================================
#  SanketVani (सङ्केतवाणी) — Continuous Sign Language Translation on iSign
#  CELL 2 / 2 : CTC MODEL TRAINING + MULTI-FORMAT PRODUCTION EXPORT
# =============================================================================
#  Reads the packed HDF5 from Cell 1 and trains `SignSentenceTranslator`
#  (Conv1D frontend -> 3-layer BiLSTM -> character CTC head), then emits the
#  three client-side deliverables:
#
#     checkpoint_sentence_model.pth   PyTorch weights (retrain / fine-tune)
#     isign_sentence_translator.onnx  opset 14, dynamic batch AND frame axes
#     vocab.json                      character map + feature contract for JS
#
#  Self-contained: this file is one notebook cell.  Run Cell 1 first.
# =============================================================================

# ------------------------------------------------------------------ 0. CONFIG
CONFIG = {
    "H5_PATH":       "/kaggle/working/isign_ctc_dataset.h5",
    "OUT_DIR":       "/kaggle/working",

    # geometry — must match Cell 1 and the JS frontend exactly
    "NUM_FRAMES":    120,
    "FEATURE_DIM":   225,

    # architecture
    "CONV_CHANNELS": 256,
    "CONV_DROPOUT":  0.2,
    "HIDDEN_DIM":    256,
    "LSTM_LAYERS":   3,
    "LSTM_DROPOUT":  0.3,
    # CTC needs several output steps per target character to have room to
    # align.  The classifier emits TIME_UPSAMPLE predictions per input frame,
    # so 192 frames become 384 CTC steps without enlarging the LSTM.
    "TIME_UPSAMPLE": 2,

    # optimisation
    "EPOCHS":        30,
    "BATCH_SIZE":    128,
    "MIN_BATCH":     8,          # OOM recovery floor
    "LR":            3e-4,
    "WEIGHT_DECAY":  1e-4,
    "WARMUP_EPOCHS": 1,
    "GRAD_CLIP":     5.0,
    "VAL_FRACTION":  0.05,
    "SEED":          1337,
    "EARLY_STOP":    8,          # epochs without val improvement (0 = off)

    # data pipeline
    "NUM_WORKERS":   4,
    "PIN_MEMORY":    True,
    "PERSISTENT":    True,
    "RDCC_NBYTES":   16 * 1024 * 1024,   # h5py chunk cache per reader

    # acceleration
    "USE_AMP":       True,       # fp16 autocast + GradScaler
    "USE_COMPILE":   "auto",     # "auto" -> only when CUDA is present
    "ONNX_OPSET":    14,
}

# --------------------------------------------------------- 1. DEPENDENCY BOOT
import importlib
import subprocess
import sys


def _ensure(module, pip_name=None, hard=True):
    try:
        return importlib.import_module(module)
    except ImportError:
        pass
    pkg = pip_name or module
    print(f"[boot] installing {pkg} ...", flush=True)
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "--no-input", pkg], check=True,
                       stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
        importlib.invalidate_caches()
        return importlib.import_module(module)
    except Exception as exc:                                   # noqa: BLE001
        if hard:
            raise RuntimeError(f"could not install required package '{pkg}'") from exc
        print(f"[boot] optional package '{pkg}' unavailable ({exc})")
        return None


np    = _ensure("numpy")
h5py  = _ensure("h5py")
torch = _ensure("torch")
_ensure("onnx", hard=False)
ort   = _ensure("onnxruntime", hard=False)
_tqdm = _ensure("tqdm", hard=False)
tqdm  = _tqdm.tqdm if _tqdm else (lambda x, **k: x)

import gc
import json
import math
import os
import random
import time

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

def _apply_env_overrides(config, prefix="SV_"):
    """Let a small paste-and-run cell override any CONFIG key via SV_<KEY>.

    Values are parsed as JSON when they can be (300, 0.5, true, null) and kept
    as a literal string otherwise, so SV_OUT_H5=/kaggle/working/x.h5 works too.
    Set a key back to its default with the string "none".
    """
    changed = []
    for key in list(config):
        raw = os.environ.get(prefix + key)
        if raw is None:
            continue
        text = raw.strip()
        if text.lower() in ("", "none", "null"):
            value = None
        else:
            try:
                value = json.loads(text)
            except ValueError:
                value = text
        config[key] = value
        changed.append(f"{key}=<hidden>" if "TOKEN" in key else f"{key}={value!r}")
    if changed:
        print("[config] overrides from environment: " + ", ".join(changed))
    return config


_apply_env_overrides(CONFIG)


# ------------------------------------------------------- 2. DETERMINISM / ENV
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(CONFIG["SEED"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")
print(f"[env] torch {torch.__version__} | device {DEVICE} | "
      f"{torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'cpu'}")

AMP_ON = bool(CONFIG["USE_AMP"]) and DEVICE.type == "cuda"


def amp_autocast(enabled=True):
    """torch.cuda.amp.autocast, routed through the non-deprecated API."""
    try:
        return torch.amp.autocast("cuda", dtype=torch.float16,
                                  enabled=enabled and AMP_ON)
    except (AttributeError, TypeError):                        # torch < 2.4
        return torch.cuda.amp.autocast(enabled=enabled and AMP_ON)


def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=AMP_ON)
    except (AttributeError, TypeError):                        # torch < 2.4
        return torch.cuda.amp.GradScaler(enabled=AMP_ON)


OOM_TYPE = getattr(torch, "OutOfMemoryError",
                   getattr(torch.cuda, "OutOfMemoryError", RuntimeError))


def is_oom(exc: BaseException) -> bool:
    return isinstance(exc, OOM_TYPE) or "out of memory" in str(exc).lower()


def free_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


# ------------------------------------------------------------- 3. VOCABULARY
BLANK_TOKEN = "<BLANK>"
LETTERS = [chr(c) for c in range(ord("a"), ord("z") + 1)]
DIGITS = [chr(c) for c in range(ord("0"), ord("9") + 1)]
PUNCT = [".", ",", "?", "!", "'"]
SPACE = [" "]
ITOS = [BLANK_TOKEN] + LETTERS + DIGITS + PUNCT + SPACE     # index 0 = blank
STOI = {ch: i for i, ch in enumerate(ITOS)}
NUM_CLASSES = len(ITOS)                                      # 43

_APOSTROPHES = {"’": "'", "ʼ": "'", "`": "'", "´": "'"}


def normalise_text(text: str) -> str:
    """Lowercase, canonicalise quotes, drop out-of-vocabulary glyphs."""
    text = "".join(_APOSTROPHES.get(c, c) for c in str(text)).lower()
    text = text.replace("–", "-").replace("—", "-")
    kept = [c if c in STOI else (" " if c.isspace() or c in "-_/;:" else "")
            for c in text]
    return " ".join("".join(kept).split())


def encode(text: str) -> np.ndarray:
    return np.asarray([STOI[c] for c in normalise_text(text) if c in STOI],
                      dtype=np.int64)


def greedy_ctc_decode(logits_row: np.ndarray) -> str:
    """argmax -> collapse repeats -> drop blanks.  Mirrors the JS decoder."""
    ids = logits_row.argmax(-1)
    out, prev = [], -1
    for i in ids:
        i = int(i)
        if i != prev and i != 0:
            out.append(ITOS[i])
        prev = i
    return "".join(out)


def char_error_rate(ref: str, hyp: str) -> float:
    if not ref:
        return float(bool(hyp))
    prev = list(range(len(hyp) + 1))
    for i, rc in enumerate(ref, 1):
        cur = [i]
        for j, hc in enumerate(hyp, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1,
                           prev[j - 1] + (rc != hc)))
        prev = cur
    return prev[-1] / len(ref)


# ----------------------------------------------------------------- 4. DATASET
class ISignCTCDataset(Dataset):
    """Lazy HDF5 reader.  The file handle is opened *inside* each worker,
    because h5py handles do not survive a fork."""

    def __init__(self, path, indices, targets):
        self.path = path
        self.indices = np.asarray(indices, dtype=np.int64)
        self.targets = targets                    # list[np.ndarray]
        self._h5 = None

    def __len__(self):
        return len(self.indices)

    def _features(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.path, "r", libver="latest",
                                 rdcc_nbytes=CONFIG["RDCC_NBYTES"],
                                 rdcc_nslots=10007)
        return self._h5["features"]

    def __getitem__(self, i):
        row = int(self.indices[i])
        x = np.asarray(self._features()[row], dtype=np.float32)
        np.nan_to_num(x, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
        return torch.from_numpy(x), torch.from_numpy(self.targets[row])

    def __del__(self):
        try:
            if self._h5 is not None:
                self._h5.close()
        except Exception:                                      # noqa: BLE001
            pass


def ctc_collate(batch):
    xs = torch.stack([b[0] for b in batch], 0)                # (B, T, F)
    ys = [b[1] for b in batch]
    target_lengths = torch.tensor([len(y) for y in ys], dtype=torch.long)
    targets = torch.cat(ys) if ys else torch.zeros(0, dtype=torch.long)
    return xs, targets, target_lengths


# ------------------------------------------------------------ 5. ARCHITECTURE
class SignSentenceTranslator(nn.Module):
    """(B, T, 225) landmarks -> (B, T*U, C) character logits for CTC.

    Conv1d layers are stride-1 / padding-1, so the temporal axis survives the
    frontend intact; the classifier then expands it by TIME_UPSAMPLE.  CTC
    input_lengths are read off the logits rather than assumed, and both axes
    stay dynamic in ONNX, so the browser may send any frame count.
    """

    def __init__(self, feature_dim=225, conv_channels=256, hidden_dim=256,
                 lstm_layers=3, num_classes=NUM_CLASSES,
                 conv_dropout=0.2, lstm_dropout=0.3, time_upsample=1):
        super().__init__()
        self.feature_dim = feature_dim
        self.num_classes = num_classes
        self.time_upsample = max(1, int(time_upsample))
        self.frontend = nn.Sequential(
            nn.Conv1d(feature_dim, conv_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(conv_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(conv_dropout),
            nn.Conv1d(conv_channels, conv_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(conv_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(conv_dropout),
        )
        self.lstm = nn.LSTM(
            input_size=conv_channels, hidden_size=hidden_dim,
            num_layers=lstm_layers, batch_first=True, bidirectional=True,
            dropout=lstm_dropout if lstm_layers > 1 else 0.0,
        )
        # One weight set per sub-step: the U predictions a frame produces are
        # independent, unlike a plain repeat which would emit U identical
        # logits and collapse straight back to one character.
        self.classifier = nn.Linear(hidden_dim * 2, num_classes * self.time_upsample)
        self._init_weights()

    def _init_weights(self):
        for name, p in self.lstm.named_parameters():
            if "weight_ih" in name:
                nn.init.xavier_uniform_(p)
            elif "weight_hh" in name:
                nn.init.orthogonal_(p)
            elif "bias" in name:
                nn.init.zeros_(p)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, x):                       # x: (B, T, F)
        x = self.frontend(x.transpose(1, 2)).transpose(1, 2)   # (B, T, C)
        x, _ = self.lstm(x)                                    # (B, T, 2H)
        logits = self.classifier(x)                            # (B, T, U*classes)
        if self.time_upsample > 1:                             # (B, T*U, classes)
            logits = logits.reshape(logits.shape[0], -1, self.num_classes)
        return logits


def ctc_objective(logits, targets, target_lengths, criterion):
    """CTC in fp32 even under autocast — fp16 log-probs overflow the alpha
    recursion on long sequences and produce NaN losses."""
    log_probs = F.log_softmax(logits.float(), dim=-1).transpose(0, 1)  # (T,B,C)
    input_lengths = torch.full((logits.shape[0],), logits.shape[1],
                               dtype=torch.long, device=logits.device)
    with torch.autocast(device_type=logits.device.type, enabled=False):
        return criterion(log_probs, targets, input_lengths, target_lengths)


# -------------------------------------------------------------- 6. DATA SPLIT
def load_index(h5_path, max_target_len):
    """Encode every transcript once, drop CTC-invalid rows, split train/val."""
    if not os.path.exists(h5_path):
        raise FileNotFoundError(
            f"{h5_path} not found — run Cell 1 (the HDF5 builder) first.")
    with h5py.File(h5_path, "r") as h5:
        if not bool(h5.attrs.get("complete", True)):
            raise RuntimeError("HDF5 is marked incomplete; re-run Cell 1.")
        if bool(h5.attrs.get("synthetic", False)):
            print("[data] WARNING: this HDF5 is the SMOKE-TEST synthetic set; "
                  "the resulting model has no linguistic value.")
        raw_texts = h5["text"][:]
        n_total = h5["features"].shape[0]
        feat_dim = h5["features"].shape[2]
    if feat_dim != CONFIG["FEATURE_DIM"]:
        raise ValueError(f"feature dim {feat_dim} != {CONFIG['FEATURE_DIM']}")

    targets, keep, dropped_empty, dropped_long = [None] * n_total, [], 0, 0
    for i in range(n_total):
        t = raw_texts[i]
        t = t.decode("utf-8", "replace") if isinstance(t, (bytes, bytearray)) else str(t)
        ids = encode(t)
        targets[i] = ids
        if len(ids) == 0:
            dropped_empty += 1
        elif len(ids) > max_target_len:
            # CTC requires target_len <= input_len; longer transcripts cannot
            # be aligned to `max_target_len` output steps.
            dropped_long += 1
        else:
            keep.append(i)
    if not keep:
        raise RuntimeError("no usable samples after transcript filtering")

    rng = np.random.default_rng(CONFIG["SEED"])
    keep = np.asarray(keep, dtype=np.int64)
    rng.shuffle(keep)
    n_val = max(1, int(round(len(keep) * CONFIG["VAL_FRACTION"])))
    n_val = min(n_val, max(1, len(keep) // 5))
    val_idx, train_idx = keep[:n_val], keep[n_val:]
    if len(train_idx) == 0:                       # tiny datasets
        train_idx, val_idx = keep, keep[:1]

    lens = [len(targets[i]) for i in keep]
    print(f"[data] {n_total} packed | usable {len(keep)} "
          f"(dropped: {dropped_empty} empty, {dropped_long} longer than "
          f"{max_target_len} chars)")
    print(f"[data] train {len(train_idx)} | val {len(val_idx)} | "
          f"target chars mean {np.mean(lens):.1f} max {max(lens)}")
    return train_idx, val_idx, targets


def make_loaders(h5_path, train_idx, val_idx, targets, batch_size):
    workers = min(CONFIG["NUM_WORKERS"], max(0, (os.cpu_count() or 2)))
    common = dict(collate_fn=ctc_collate, num_workers=workers,
                  pin_memory=CONFIG["PIN_MEMORY"] and DEVICE.type == "cuda")
    if workers > 0:
        common.update(persistent_workers=CONFIG["PERSISTENT"],
                      prefetch_factor=4)
    train = DataLoader(ISignCTCDataset(h5_path, train_idx, targets),
                       batch_size=batch_size, shuffle=True, drop_last=False,
                       **common)
    val = DataLoader(ISignCTCDataset(h5_path, val_idx, targets),
                     batch_size=max(1, batch_size // 2), shuffle=False,
                     drop_last=False, **common)
    return train, val


# ------------------------------------------------------------ 7. TRAIN / EVAL
def build_model():
    model = SignSentenceTranslator(
        feature_dim=CONFIG["FEATURE_DIM"],
        conv_channels=CONFIG["CONV_CHANNELS"],
        hidden_dim=CONFIG["HIDDEN_DIM"],
        lstm_layers=CONFIG["LSTM_LAYERS"],
        num_classes=NUM_CLASSES,
        conv_dropout=CONFIG["CONV_DROPOUT"],
        lstm_dropout=CONFIG["LSTM_DROPOUT"],
        time_upsample=CONFIG["TIME_UPSAMPLE"],
    ).to(DEVICE)
    n = sum(p.numel() for p in model.parameters())
    print(f"[model] SignSentenceTranslator | {n / 1e6:.2f}M parameters | "
          f"{NUM_CLASSES} classes")
    return model


def maybe_compile(model):
    """PyTorch 2.x kernel fusion; silently degrades to eager if unsupported."""
    want = CONFIG["USE_COMPILE"]
    if want == "auto":
        want = DEVICE.type == "cuda"
    if not want or not hasattr(torch, "compile"):
        return model, False
    try:
        compiled = torch.compile(model)
        print("[accel] torch.compile enabled")
        return compiled, True
    except Exception as exc:                                   # noqa: BLE001
        print(f"[accel] torch.compile unavailable ({exc}); running eager")
        return model, False


def base_module(model):
    """Unwrap torch.compile / DataParallel for saving and exporting."""
    return getattr(model, "_orig_mod", getattr(model, "module", model))


def run_epoch(model, loader, criterion, optimiser, scaler, epoch, epochs):
    model.train()
    total, seen = 0.0, 0
    bar = tqdm(loader, desc=f"epoch {epoch + 1}/{epochs}", leave=False)
    for xs, targets, target_lengths in bar:
        xs = xs.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        target_lengths = target_lengths.to(DEVICE, non_blocking=True)
        optimiser.zero_grad(set_to_none=True)
        with amp_autocast():
            logits = model(xs)
        loss = ctc_objective(logits, targets, target_lengths, criterion)
        if not torch.isfinite(loss):
            continue                       # degenerate batch: skip, keep going
        scaler.scale(loss).backward()
        scaler.unscale_(optimiser)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["GRAD_CLIP"])
        scaler.step(optimiser)
        scaler.update()
        bs = xs.shape[0]
        total += float(loss.detach()) * bs
        seen += bs
        if hasattr(bar, "set_postfix"):
            bar.set_postfix(loss=f"{total / max(seen, 1):.4f}")
    return total / max(seen, 1)


@torch.no_grad()
def evaluate(model, loader, criterion, show=3):
    model.eval()
    total, seen, cers, samples = 0.0, 0, [], []
    for xs, targets, target_lengths in loader:
        xs = xs.to(DEVICE, non_blocking=True)
        tg = targets.to(DEVICE, non_blocking=True)
        tl = target_lengths.to(DEVICE, non_blocking=True)
        with amp_autocast():
            logits = model(xs)
        loss = ctc_objective(logits, tg, tl, criterion)
        if torch.isfinite(loss):
            total += float(loss) * xs.shape[0]
            seen += xs.shape[0]
        probs = logits.float().cpu().numpy()
        cursor = 0
        for b in range(xs.shape[0]):
            n = int(target_lengths[b])
            ref = "".join(ITOS[int(i)] for i in targets[cursor:cursor + n])
            cursor += n
            hyp = greedy_ctc_decode(probs[b])
            cers.append(char_error_rate(ref, hyp))
            if len(samples) < show:
                samples.append((ref, hyp))
    return (total / max(seen, 1), float(np.mean(cers)) if cers else 1.0, samples)


def train(h5_path):
    # T_out is measured, not assumed, so CTC lengths stay correct if the
    # frontend stack ever changes stride.
    probe = SignSentenceTranslator(feature_dim=CONFIG["FEATURE_DIM"],
                                   conv_channels=CONFIG["CONV_CHANNELS"],
                                   hidden_dim=CONFIG["HIDDEN_DIM"],
                                   lstm_layers=CONFIG["LSTM_LAYERS"],
                                   time_upsample=CONFIG["TIME_UPSAMPLE"])
    probe.eval()
    with torch.no_grad():
        t_out = probe(torch.zeros(1, CONFIG["NUM_FRAMES"],
                                  CONFIG["FEATURE_DIM"])).shape[1]
    del probe
    gc.collect()
    print(f"[ctc] {CONFIG['NUM_FRAMES']} input frames -> {t_out} output steps")

    train_idx, val_idx, targets = load_index(h5_path, t_out)
    model = build_model()
    model, compiled = maybe_compile(model)
    criterion = nn.CTCLoss(blank=0, zero_infinity=True)
    optimiser = torch.optim.AdamW(model.parameters(), lr=CONFIG["LR"],
                                  weight_decay=CONFIG["WEIGHT_DECAY"])
    scaler = make_scaler()

    batch_size = CONFIG["BATCH_SIZE"]
    train_loader, val_loader = make_loaders(h5_path, train_idx, val_idx,
                                            targets, batch_size)
    print(f"[train] batch {batch_size} | workers {train_loader.num_workers} "
          f"| AMP {'on' if AMP_ON else 'off'}")

    ckpt_path = os.path.join(CONFIG["OUT_DIR"], "checkpoint_sentence_model.pth")
    best, stale, history = math.inf, 0, []
    epochs = CONFIG["EPOCHS"]

    def lr_at(ep):
        warm = max(1, CONFIG["WARMUP_EPOCHS"])
        if ep < warm:
            return CONFIG["LR"] * (ep + 1) / warm
        p = (ep - warm) / max(1, epochs - warm)
        return CONFIG["LR"] * 0.5 * (1 + math.cos(math.pi * min(p, 1.0)))

    for epoch in range(epochs):
        for g in optimiser.param_groups:
            g["lr"] = lr_at(epoch)
        t0 = time.time()

        # ---- dynamic OOM recovery: halve the batch and retry the epoch ----
        while True:
            try:
                tr_loss = run_epoch(model, train_loader, criterion, optimiser,
                                    scaler, epoch, epochs)
                va_loss, va_cer, samples = evaluate(model, val_loader, criterion)
                break
            except Exception as exc:                           # noqa: BLE001
                if not is_oom(exc) or batch_size <= CONFIG["MIN_BATCH"]:
                    raise
                new_bs = max(CONFIG["MIN_BATCH"], batch_size // 2)
                print(f"\n[oom] CUDA out of memory at batch {batch_size} -> "
                      f"retrying this epoch at {new_bs}")
                optimiser.zero_grad(set_to_none=True)
                del train_loader, val_loader
                free_cuda()
                batch_size = new_bs
                train_loader, val_loader = make_loaders(
                    h5_path, train_idx, val_idx, targets, batch_size)
                scaler = make_scaler()

        history.append({"epoch": epoch + 1, "train_loss": tr_loss,
                        "val_loss": va_loss, "val_cer": va_cer,
                        "lr": lr_at(epoch), "batch_size": batch_size})
        print(f"[epoch {epoch + 1:>3}/{epochs}] train {tr_loss:.4f} | "
              f"val {va_loss:.4f} | CER {va_cer:.3f} | "
              f"lr {lr_at(epoch):.2e} | {time.time() - t0:.1f}s")
        for ref, hyp in samples[:2]:
            print(f"           ref: {ref[:70]!r}\n           hyp: {hyp[:70]!r}")

        if va_loss < best - 1e-4:
            best, stale = va_loss, 0
            torch.save({
                "model_state_dict": base_module(model).state_dict(),
                "epoch": epoch + 1, "val_loss": va_loss, "val_cer": va_cer,
                "vocab": ITOS, "num_classes": NUM_CLASSES,
                "config": {k: CONFIG[k] for k in
                           ("NUM_FRAMES", "FEATURE_DIM", "CONV_CHANNELS",
                            "CONV_DROPOUT", "HIDDEN_DIM", "LSTM_LAYERS",
                            "LSTM_DROPOUT", "TIME_UPSAMPLE")},
            }, ckpt_path)
            print(f"           saved -> {ckpt_path}")
        else:
            stale += 1
            if CONFIG["EARLY_STOP"] and stale >= CONFIG["EARLY_STOP"]:
                print(f"[train] no improvement for {stale} epochs — stopping")
                break

        free_cuda()          # active memory management between epochs

    if os.path.exists(ckpt_path):
        state = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        base_module(model).load_state_dict(state["model_state_dict"])
        print(f"[train] restored best checkpoint (val {state['val_loss']:.4f})")
    return base_module(model), history


# ---------------------------------------------------------------- 8. EXPORTS
def export_vocab(path):
    payload = {
        "name": "SanketVani", "task": "continuous-sign-language-translation",
        "blank_token": BLANK_TOKEN, "blank_index": 0,
        "num_classes": NUM_CLASSES,
        "itos": ITOS,
        "stoi": STOI,
        "input": {
            "num_frames": CONFIG["NUM_FRAMES"],
            "feature_dim": CONFIG["FEATURE_DIM"],
            "dtype": "float32",
            "shape": ["batch_size", "num_frames", CONFIG["FEATURE_DIM"]],
            "coord_space": "mediapipe_normalized (x,y in [0,1]; z on the x scale)",
            "missing_value": 0.0,
            "resample": "uniform: idx = round(linspace(0, n-1, num_frames))",
            "layout": [
                {"name": "pose",       "landmarks": 33, "slice": [0, 99],
                 "source": "Holistic results.poseLandmarks"},
                {"name": "left_hand",  "landmarks": 21, "slice": [99, 162],
                 "source": "Holistic results.leftHandLandmarks"},
                {"name": "right_hand", "landmarks": 21, "slice": [162, 225],
                 "source": "Holistic results.rightHandLandmarks"},
            ],
            "order": "per landmark: x, y, z (contiguous, landmark-major)",
        },
        "output": {"name": "logits",
                   "shape": ["batch_size",
                             f"num_frames * {CONFIG['TIME_UPSAMPLE']}",
                             NUM_CLASSES],
                   "time_upsample": CONFIG["TIME_UPSAMPLE"],
                   "decode": "argmax over the last axis, collapse consecutive "
                             "duplicates, then drop index 0 (<BLANK>). Read the "
                             "step count from the output tensor, not from "
                             "num_frames."},
    }
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(payload, fh, ensure_ascii=False, indent=2)
    print(f"[export] vocab.json -> {path}")


def export_onnx(model, path):
    """opset-14 export with dynamic batch AND frame axes."""
    model = model.eval().to("cpu").float()
    # batch 1 at trace time (the exporter's own recommendation for LSTM
    # graphs); the batch axis is still dynamic and is proven so by verify_onnx.
    dummy = torch.randn(1, CONFIG["NUM_FRAMES"], CONFIG["FEATURE_DIM"])
    dynamic_axes = {"landmarks": {0: "batch_size", 1: "num_frames"},
                    "logits": {0: "batch_size", 1: "out_steps"}}
    common = dict(input_names=["landmarks"], output_names=["logits"],
                  dynamic_axes=dynamic_axes, opset_version=CONFIG["ONNX_OPSET"],
                  do_constant_folding=True)
    attempts = [
        # TorchScript exporter: the predictable path for LSTM graphs.
        lambda: torch.onnx.export(model, (dummy,), path, dynamo=False, **common),
        lambda: torch.onnx.export(model, (dummy,), path, **common),
        lambda: torch.onnx.export(model, (dummy,), path, dynamo=True, **common),
    ]
    last = None
    for i, attempt in enumerate(attempts, 1):
        try:
            with torch.no_grad():
                attempt()
            if os.path.exists(path) and os.path.getsize(path) > 0:
                print(f"[export] ONNX (strategy {i}, opset "
                      f"{CONFIG['ONNX_OPSET']}) -> {path} "
                      f"({os.path.getsize(path) / 2**20:.1f} MiB)")
                return True
        except Exception as exc:                               # noqa: BLE001
            last = exc
            print(f"[export] ONNX strategy {i} failed: "
                  f"{type(exc).__name__}: {str(exc)[:180]}")
    raise RuntimeError(f"ONNX export failed on every strategy: {last}")


def verify_onnx(model, path):
    """Load the exported graph in onnxruntime and prove it matches PyTorch."""
    try:
        import onnx
        onnx.checker.check_model(onnx.load(path))
        print("[verify] onnx.checker: graph is well-formed")
    except Exception as exc:                                   # noqa: BLE001
        print(f"[verify] onnx.checker unavailable/failed: {exc}")
    if ort is None:
        print("[verify] onnxruntime missing — skipped the runtime check")
        return False

    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    in_name = sess.get_inputs()[0].name
    model = model.eval().to("cpu").float()
    ok = True
    # exercise both dynamic axes: the training shape and an odd live-webcam one
    for batch, frames in ((1, CONFIG["NUM_FRAMES"]), (3, 97)):
        x = np.random.randn(batch, frames, CONFIG["FEATURE_DIM"]).astype(np.float32)
        out = sess.run(None, {in_name: x})[0]
        with torch.no_grad():
            ref = model(torch.from_numpy(x)).numpy()
        steps = frames * CONFIG["TIME_UPSAMPLE"]
        assert out.shape == (batch, steps, NUM_CLASSES), \
            f"onnx returned {out.shape}, expected {(batch, steps, NUM_CLASSES)}"
        diff = float(np.abs(out - ref).max())
        agree = (out.argmax(-1) == ref.argmax(-1)).mean()
        ok &= diff < 2e-3
        print(f"[verify] ({batch}, {frames}, {CONFIG['FEATURE_DIM']}) -> "
              f"{out.shape} | max|onnx-torch| {diff:.2e} | argmax agreement "
              f"{agree * 100:.1f}%")
    # end-to-end decode on the ONNX output, exactly as the browser will do it
    x = np.random.randn(1, CONFIG["NUM_FRAMES"], CONFIG["FEATURE_DIM"]).astype(np.float32)
    print(f"[verify] dummy greedy decode: "
          f"{greedy_ctc_decode(sess.run(None, {in_name: x})[0][0])!r}")
    print(f"[verify] ONNX graph is {'VALID' if ok else 'DIVERGENT'}")
    return ok


# -------------------------------------------------------------------- 9. MAIN
def main():
    os.makedirs(CONFIG["OUT_DIR"], exist_ok=True)
    t0 = time.time()
    model, history = train(CONFIG["H5_PATH"])

    ckpt = os.path.join(CONFIG["OUT_DIR"], "checkpoint_sentence_model.pth")
    onnx_path = os.path.join(CONFIG["OUT_DIR"], "isign_sentence_translator.onnx")
    vocab_path = os.path.join(CONFIG["OUT_DIR"], "vocab.json")

    export_vocab(vocab_path)
    export_onnx(model, onnx_path)
    verify_onnx(model, onnx_path)

    with open(os.path.join(CONFIG["OUT_DIR"], "training_history.json"), "w") as fh:
        json.dump(history, fh, indent=2)

    free_cuda()
    print("\n=== SanketVani deliverables " + "=" * 40)
    for p in (ckpt, onnx_path, vocab_path):
        flag = "OK " if os.path.exists(p) else "MISSING"
        size = f"{os.path.getsize(p) / 2**20:6.2f} MiB" if os.path.exists(p) else "-"
        print(f"  [{flag}] {size}  {p}")
    print(f"=== total wall time {time.time() - t0:.1f}s " + "=" * 30)

main()


[boot] installing onnxruntime ...
[config] overrides from environment: NUM_FRAMES=192, TIME_UPSAMPLE=2, EPOCHS=30
[env] torch 2.10.0+cu128 | device cuda | Tesla T4
[ctc] 192 input frames -> 384 output steps
[data] 123119 packed | usable 123116 (dropped: 2 empty, 1 longer than 384 chars)
[data] train 116960 | val 6156 | target chars mean 59.8 max 352
[model] SignSentenceTranslator | 4.62M parameters | 43 classes
[accel] torch.compile enabled
[train] batch 128 | workers 4 | AMP on


epoch 1/30:   0%|          | 0/914 [00:00<?, ?it/s]W0906 14:30:53.861000 22 torch/_inductor/utils.py:1679] [0/0_1] Not enough SMs to use max_autotune_gemm mode


[epoch   1/30] train 3.5979 | val 3.1766 | CER 0.990 | lr 3.00e-04 | 108.4s
           ref: 'in his letter he said'
           hyp: ''
           ref: 'to look for different angles in their class.'
           hyp: ''
           saved -> /kaggle/working/checkpoint_sentence_model.pth


[epoch   2/30] train 3.1282 | val 3.2508 | CER 0.989 | lr 3.00e-04 | 89.0s
           ref: 'in his letter he said'
           hyp: '.'
           ref: 'to look for different angles in their class.'
           hyp: ''


[epoch   3/30] train 3.0727 | val 3.0186 | CER 0.891 | lr 2.99e-04 | 88.7s
           ref: 'in his letter he said'
           hyp: '   .'
           ref: 'to look for different angles in their class.'
           hyp: ''
           saved -> /kaggle/working/checkpoint_sentence_model.pth


...

[epoch  28/30] train 2.9061 | val 2.9003 | CER 0.843 | lr 7.85e-06 | 90.4s
           ref: 'in his letter he said'
           hyp: 'e  e   .'
           ref: 'to look for different angles in their class.'
           hyp: '.'
           saved -> /kaggle/working/checkpoint_sentence_model.pth


[epoch  29/30] train 2.9054 | val 2.8999 | CER 0.844 | lr 3.51e-06 | 89.6s
           ref: 'in his letter he said'
           hyp: 'e  e   .'
           ref: 'to look for different angles in their class.'
           hyp: '.'
           saved -> /kaggle/working/checkpoint_sentence_model.pth


[epoch  30/30] train 2.9051 | val 2.9001 | CER 0.843 | lr 8.79e-07 | 91.5s
           ref: 'in his letter he said'
           hyp: 'e  e   .'
           ref: 'to look for different angles in their class.'
           hyp: '.'
[train] restored best checkpoint (val 2.8999)
[export] vocab.json -> /kaggle/working/vocab.json


/tmp/ipykernel_22/552133191.py:666: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  lambda: torch.onnx.export(model, (dummy,), path, dynamo=False, **common),
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:4445: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  return _generic_rnn(


[export] ONNX (strategy 1, opset 14) -> /kaggle/working/isign_sentence_translator.onnx (17.6 MiB)
[verify] onnx.checker: graph is well-formed
[verify] (1, 192, 225) -> (1, 384, 43) | max|onnx-torch| 2.86e-06 | argmax agreement 100.0%
[verify] (3, 97, 225) -> (3, 194, 43) | max|onnx-torch| 3.46e-06 | argmax agreement 100.0%
[verify] dummy greedy decode: '  .'
[verify] ONNX graph is VALID

=== SanketVani deliverables ========================================
  [OK ]  17.64 MiB  /kaggle/working/checkpoint_sentence_model.pth
  [OK ]  17.63 MiB  /kaggle/working/isign_sentence_translator.onnx
  [OK ]   0.00 MiB  /kaggle/working/vocab.json
=== total wall time 2733.6s ==============================
